In [6]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime

pd.set_option('display.max_rows', 500)


In [7]:
# ── 1. PATH CONFIGURATION ───────────────
BASE_DATA_DIR = "../../../data"

RAW_DIR    = os.path.join(BASE_DATA_DIR, "raw")
BRONZE_DIR = os.path.join(BASE_DATA_DIR, "bronze")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(BRONZE_DIR, exist_ok=True)

CSV_URL = "https://www.data.gouv.fr/api/1/datasets/r/2b27a675-e3bf-41ef-a852-5fb9ab483967"

path_csv_raw         = os.path.join(RAW_DIR, "DEP-departementale.csv")
path_departements_bronze = os.path.join(BRONZE_DIR, "bronze_DEP-departementale.csv")


In [8]:
# ── 2. RAW LAYER: Landing Zone ──────────────────────────────────────────────
# We keep the source file exactly as it is.
if not os.path.exists(path_csv_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(CSV_URL, timeout=180)
    resp.raise_for_status()
    with open(path_csv_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_csv_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


✅ Raw file already exists in ../../../data/raw


In [9]:
# ── 3. BRONZE LAYER: Ingestion & Metadata ───────────────────────────────────
# Bronze is for "raw-ish" data in a readable format with audit columns.
print("\n🛠 Processing RAW -> BRONZE...")
df_bronze = pd.read_csv(path_csv_raw, sep=";", engine="python", dtype=str)

# Normalize column names by trimming surrounding spaces
df_bronze.columns = [c.strip() if isinstance(c, str) else c for c in df_bronze.columns]
df_bronze = df_bronze[df_bronze["Code_departement"] == "69"]
df_bronze['extraction_source_url'] = CSV_URL
df_bronze['ingestion_timestamp']   = datetime.now().isoformat()
df_bronze['source_file_name']      = os.path.basename(path_csv_raw)

df_bronze.to_csv(path_departements_bronze, index=False, sep=";", encoding="utf-8")

print(f"✅ BRONZE dataset created: {path_departements_bronze}")
print(f"📊 Rows: {len(df_bronze)} | Columns: {len(df_bronze.columns)}")



🛠 Processing RAW -> BRONZE...
✅ BRONZE dataset created: ../../../data/bronze/bronze_DEP-departementale.csv
📊 Rows: 180 | Columns: 14


In [10]:
df_bronze.head

<bound method NDFrame.head of       Code_departement Code_region annee  \
69                  69          84  2016   
170                 69          84  2017   
271                 69          84  2018   
372                 69          84  2019   
473                 69          84  2020   
574                 69          84  2021   
675                 69          84  2022   
776                 69          84  2023   
877                 69          84  2024   
978                 69          84  2025   
1079                69          84  2016   
1180                69          84  2017   
1281                69          84  2018   
1382                69          84  2019   
1483                69          84  2020   
1584                69          84  2021   
1685                69          84  2022   
1786                69          84  2023   
1887                69          84  2024   
1988                69          84  2025   
2089                69          84  2016   
21